# 01 · idealista — Valoración automática de inmuebles (regresión)

**Problema:** estimar el precio de una vivienda a partir de sus características, como hace un AVM (*Automated Valuation Model*).

**Qué vas a ver paso a paso**

1. Generar un mercado sintético de 480 viviendas con una «verdad» conocida.
2. Separar entrenamiento y prueba.
3. Ajustar una regresión lineal y comparar los coeficientes aprendidos con los reales.
4. Valorar por **comparables** (las 6 viviendas más parecidas de la misma zona).
5. Mezclar ambos métodos y medir el error (MAE) con datos que el modelo no ha visto.
6. **Tasador interactivo:** cambia la vivienda y observa en tiempo real cómo se forma el precio.

> Línea base didáctica independiente con datos sintéticos (semilla 42). **No es el AVM de idealista ni usa sus datos.** Reproduce el cálculo de `general/es/tools/generate-idealista-demo.py`.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown

import innoit_style as inno
from innoit_style import MORADO, CORAL, TEAL, LAVANDA, LAVANDA_OSCURA, GRIS, TEXTO_SUAVE, PALETA, cascada
inno.aplicar()   # colores, tipografía Urbane y mapas de color de InnoIT

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

ZONAS = ['Zona A', 'Zona B', 'Zona C']
COLOR_ZONA = [TEAL, MORADO, CORAL]

## 1 · Generar el mercado sintético

Cada vivienda tiene superficie, dormitorios, zona (A, B o C), si está reformada y si tiene ascensor. El precio «real» sigue una fórmula conocida más ruido aleatorio (σ = 12.000 €):

$$\text{precio} = 20.000 + 2.000\cdot m^2 + 12.000\cdot \text{dorm} + 60.000\cdot[B] + 120.000\cdot[C] + 25.000\cdot \text{reformado} + 15.000\cdot \text{ascensor} + \varepsilon$$

El modelo **no conoce** esta fórmula: tiene que descubrirla a partir de los ejemplos.

In [ ]:
rng = np.random.default_rng(42)
n = 480
area = rng.integers(45, 141, n)
beds = rng.integers(1, 5, n)
zone = rng.integers(0, 3, n)
renovated = rng.integers(0, 2, n)
lift = rng.integers(0, 2, n)
X = np.column_stack([area, beds, zone == 1, zone == 2, renovated, lift]).astype(float)
REAL = dict(intercepto=20000, area=2000, beds=12000, zoneB=60000, zoneC=120000, renovated=25000, lift=15000)
y = np.rint(20000 + 2000*area + 12000*beds + 60000*(zone == 1) + 120000*(zone == 2)
            + 25000*renovated + 15000*lift + rng.normal(0, 12000, n))

df = pd.DataFrame(dict(m2=area, dormitorios=beds, zona=[ZONAS[z] for z in zone],
                       reformado=renovated.astype(bool), ascensor=lift.astype(bool), precio=y.astype(int)))
display(df.head(8))
df.describe().round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
for z in range(3):
    m = zone == z
    ax.scatter(area[m], y[m] / 1000, s=10 + 8*beds[m], alpha=.6, color=COLOR_ZONA[z], label=ZONAS[z])
ax.set(xlabel='Superficie (m²)', ylabel='Precio (miles de €)', title='Mercado sintético · tamaño del punto = dormitorios')
ax.legend();

## 2 · Separar entrenamiento y prueba

El 75 % de las viviendas se usa para aprender; el 25 % se guarda para medir el error con casas que el modelo **no ha visto**.

In [ ]:
train, test = train_test_split(np.arange(n), test_size=.25, random_state=42)
print(f'Entrenamiento: {len(train)} viviendas · Prueba: {len(test)} viviendas')

## 3 · Regresión lineal

La regresión busca los pesos que minimizan el error cuadrático. Como conocemos la fórmula real, podemos comprobar qué tan bien la ha recuperado.

In [ ]:
model = LinearRegression().fit(X[train], y[train])
NOMBRES = ['m²', 'Dormitorio', 'Zona B', 'Zona C', 'Reformado', 'Ascensor']
tabla = pd.DataFrame({'Real': list(REAL.values()),
                      'Aprendido': [model.intercept_, *model.coef_]},
                     index=['Intercepto', *NOMBRES]).round(0)
tabla['Diferencia'] = tabla['Aprendido'] - tabla['Real']
tabla

### Aprendizaje paso a paso (descenso de gradiente)

`LinearRegression` resuelve el problema de golpe. Para ver **cómo** se aprende, entrenamos el mismo modelo con descenso de gradiente sobre variables estandarizadas y guardamos cada iteración. Pulsa ▶ para reproducirlo.

In [ ]:
mu, sd = X[train].mean(0), X[train].std(0)
Xs = (X[train] - mu) / sd
w, b, lr, hist = np.zeros(6), y[train].mean(), .02, []
for it in range(301):
    pred = Xs @ w + b
    err = pred - y[train]
    if it % 5 == 0:
        hist.append((it, (w / sd).copy(), b - (w * mu / sd).sum(), np.abs(err).mean()))
    w -= lr * Xs.T @ err / len(train)
    b -= lr * err.mean()

def dibujar_gd(paso):
    it, coef, icpt, mae = hist[paso]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw=dict(width_ratios=[1.3, 1]))
    xs = np.arange(len(NOMBRES))
    a1.bar(xs - .2, list(REAL.values())[1:], .4, color=GRIS, label='Real')
    a1.bar(xs + .2, coef, .4, color=MORADO, label='Aprendido')
    a1.set_xticks(xs, NOMBRES); a1.set_title(f'Coeficientes · iteración {it}'); a1.legend()
    a2.plot([h[0] for h in hist], [h[3] / 1000 for h in hist], color=GRIS)
    a2.scatter([it], [mae / 1000], color=CORAL, zorder=3)
    a2.set(title=f'Error medio en entrenamiento: {mae:,.0f} €', xlabel='Iteración', ylabel='MAE (miles €)')
    plt.tight_layout(); plt.show()

play = widgets.Play(min=0, max=len(hist) - 1, interval=150)
slider = widgets.IntSlider(min=0, max=len(hist) - 1, description='Paso')
widgets.jslink((play, 'value'), (slider, 'value'))
display(widgets.HBox([play, slider]), widgets.interactive_output(dibujar_gd, {'paso': slider}))

## 4 · Valoración por comparables

Un tasador humano busca viviendas parecidas **en la misma zona**. La distancia combina diferencia de superficie (escala 30 m²), de dormitorios (escala 2) y una penalización de 0,25 si cambia la reforma o el ascensor. El precio es la media de los 6 más cercanos.

In [ ]:
tr = dict(area=area[train], beds=beds[train], zone=zone[train], ren=renovated[train], lift=lift[train], price=y[train])

def comparables(a, d, z, r, l, k=6):
    """Devuelve los índices (dentro de train) de las k viviendas más parecidas de la misma zona."""
    pool = np.where(tr['zone'] == z)[0]
    dist = (((tr['area'][pool] - a) / 30)**2 + ((tr['beds'][pool] - d) / 2)**2
            + .25*(tr['ren'][pool] != r) + .25*(tr['lift'][pool] != l))
    orden = np.lexsort((train[pool], dist))      # desempate por id, igual que el script original
    return pool[orden[:k]]

def precio_comparables(i):
    return tr['price'][comparables(area[i], beds[i], zone[i], renovated[i], lift[i])].mean()

pred_reg = model.predict(X[test])
pred_comp = np.array([precio_comparables(i) for i in test])
print('Ejemplo · vivienda de prueba', test[0] + 1)
display(df.iloc[[test[0]]])
display(df.iloc[train[comparables(area[test[0]], beds[test[0]], zone[test[0]], renovated[test[0]], lift[test[0]])]])

## 5 · Mezcla y error con datos no vistos

Combinamos `peso × regresión + (1 − peso) × comparables`. Mueve el peso y observa cómo cambia el MAE en el conjunto de prueba.

In [ ]:
pesos = np.linspace(0, 1, 101)
maes = np.array([mean_absolute_error(y[test], w*pred_reg + (1 - w)*pred_comp) for w in pesos])
print(f'MAE regresión: {mean_absolute_error(y[test], pred_reg):,.0f} € · '
      f'comparables: {mean_absolute_error(y[test], pred_comp):,.0f} € · '
      f'mezcla 50/50: {mean_absolute_error(y[test], .5*pred_reg + .5*pred_comp):,.0f} €')

def mezcla(peso=.5):
    p = peso*pred_reg + (1 - peso)*pred_comp
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
    a1.plot(pesos, maes / 1000, color=MORADO)
    a1.scatter([peso], [mean_absolute_error(y[test], p) / 1000], color=CORAL, zorder=3, s=50)
    a1.axvline(pesos[maes.argmin()], ls='--', color=GRIS)
    a1.set(xlabel='Peso de la regresión', ylabel='MAE (miles €)',
           title=f'MAE = {mean_absolute_error(y[test], p):,.0f} € · óptimo en {pesos[maes.argmin()]:.2f}')
    a2.scatter(y[test] / 1000, p / 1000, s=12, alpha=.6, color=[COLOR_ZONA[z] for z in zone[test]])
    lim = [y.min() / 1000, y.max() / 1000]
    a2.plot(lim, lim, color=MORADO, lw=.8)
    a2.set(xlabel='Precio real (miles €)', ylabel='Precio estimado (miles €)', title='Real vs estimado (prueba)')
    plt.tight_layout(); plt.show()

widgets.interact(mezcla, peso=widgets.FloatSlider(value=.5, min=0, max=1, step=.05, description='Peso'));

## 6 · Tasador interactivo

Configura una vivienda. A la izquierda, la **cascada** muestra cómo la regresión parte del precio medio y suma o resta cada característica: `coeficiente × (valor − media de entrenamiento)`. A la derecha, las viviendas de su zona y los 6 comparables elegidos.

In [ ]:
medias = X[train].mean(0)
base = float(model.predict(X[train]).mean())

def tasar(m2=90, dormitorios=2, zona='Zona B', reformado=True, ascensor=False, peso=.5):
    z = ZONAS.index(zona)
    x = np.array([m2, dormitorios, z == 1, z == 2, reformado, ascensor], float)
    p_reg = float(model.predict(x[None])[0])
    idx = comparables(m2, dormitorios, z, int(reformado), int(ascensor))
    p_comp = float(tr['price'][idx].mean())
    p_final = peso*p_reg + (1 - peso)*p_comp

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2), gridspec_kw=dict(width_ratios=[1.1, 1]))
    cascada(a1, base / 1000, NOMBRES, model.coef_ * (x - medias) / 1000, 'Media', 'Regresión', '{:,.1f}')
    a1.set(ylabel='Miles de €', title='Cómo se forma el precio de la regresión')
    m = tr['zone'] == z
    a2.scatter(tr['area'][m], tr['price'][m] / 1000, s=12, color=GRIS, alpha=.5, label=f'Viviendas en {zona}')
    a2.scatter(tr['area'][idx], tr['price'][idx] / 1000, s=60, facecolor='none', edgecolor=TEAL, lw=2, label='6 comparables')
    a2.scatter([m2], [p_final / 1000], marker='*', s=260, color=CORAL, zorder=4, label='Tu vivienda (mezcla)')
    a2.set(xlabel='m²', ylabel='Miles de €', title='Comparables en la misma zona'); a2.legend(fontsize=8)
    plt.tight_layout(); plt.show()
    display(Markdown(f'**Regresión:** {p_reg:,.0f} € · **Comparables:** {p_comp:,.0f} € · '
                     f'**Valoración final ({peso:.0%} regresión):** {p_final:,.0f} €'))

widgets.interact(tasar,
    m2=widgets.IntSlider(90, 45, 140, description='m²'),
    dormitorios=widgets.IntSlider(2, 1, 4, description='Dormitorios'),
    zona=widgets.ToggleButtons(options=ZONAS, description='Zona'),
    reformado=widgets.Checkbox(True, description='Reformado'),
    ascensor=widgets.Checkbox(False, description='Ascensor'),
    peso=widgets.FloatSlider(value=.5, min=0, max=1, step=.05, description='Peso regr.'));

## Para llevarse

- La regresión **generaliza** (aprende cuánto vale cada m² en todo el mercado); los comparables capturan **lo local**.
- Mezclarlos reduce el error porque sus fallos no son los mismos.
- La cascada hace el modelo **explicable**: cada euro del precio tiene una causa.